## 1. Import Requirements

# Master Shear-Wave Splitting Workflow for Axial Seamount

This notebook provides a complete, clean workflow from raw earthquake catalog and waveform data to shear-wave splitting analysis results. The workflow follows proper sequencing and includes all necessary quality control measures. 

Instead of using catalog from Wilcock and Zhang or ML DD, we use the nlloc file for all stations from Christian's results.

## Workflow Overview

1. **Data Loading & Initial Setup** - Load earthquake catalog and station metadata
2. **Extended Time Window Creation** - Create proper time windows for waveform retrieval
3. **Waveform Data Retrieval** - Download seismic data with extended windows
4. **Quality Control Filters** - P-wave rectilinearity, SNR, and incidence angle filtering
5. **Geometric Calculations** - Back-azimuth and distance calculations
6. **Shear-Wave Splitting Analysis** - Dynamic parameter estimation and SWSPy analysis
7. **Results Processing & Visualization** - Compile and visualize splitting parameters

## Key Improvements
- Extended catalog creation moved to proper early position
- Updated P-wave polarization analysis for true incidence angles
- Integrated SNR calculations with proper S-wave timing
- Clean separation of quality control steps

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load 2018 earthquake test catalog - ML DD
#catalog = pd.read_csv('2018_eq_catalog.csv')

# Load Baillard nonlinloc catalog
catalog = pd.read_csv('AXIAL.PHASE.FINAL_3D_V2.csv')

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:
# Filter catalog for AXAS2 station only
#axas2_catalog = catalog[catalog['station'] == 'OOAXAS2'].copy()

axas2_catalog = catalog[catalog['station'] == 'AXAS2'].copy()
#axec2_catalog = catalog[catalog['station'] == 'AXEC2'].copy()

# Reset index to ensure clean indexing
axas2_catalog = axas2_catalog.reset_index(drop=True)
#axec2_catalog = axec2_catalog.reset_index(drop=True)

# Pick all events from April and May, 2015
axas2_catalog['datetime'] = pd.to_datetime(axas2_catalog['datetime'])
axas2_catalog = axas2_catalog[(axas2_catalog['datetime'] >= '2015-04-20') & (axas2_catalog['datetime'] < '2015-04-28')].copy()

#axec2_catalog['datetime'] = pd.to_datetime(axec2_catalog['datetime'])
#axec2_catalog = axec2_catalog[(axec2_catalog['datetime'] >= '2015-04-20') & (axec2_catalog['datetime'] < '2015-04-28')].copy()

# Select first 100 events for testing
#test_catalog_100 = axas2_catalog.head(100).copy()
print(f"Total AXAS2 events in catalog: {len(axas2_catalog)}")
#print(f"Total AXEC2 events in catalog: {len(axec2_catalog)}")

#print(f"Test catalog created with first {len(test_catalog_100)} AXAS2 events")
print(f"\nDate range of test catalog:")
print(f"Start: {axas2_catalog['datetime'].min()}")
print(f"End: {axas2_catalog['datetime'].max()}")
#print(f"Start: {axec2_catalog['datetime'].min()}")
#print(f"End: {axec2_catalog['datetime'].max()}")

display(axas2_catalog)
#display(axec2_catalog)

In [ ]:
# Downsample catalog by a factor of 4 - take every fourth event
#test_catalog = axas2_catalog.iloc[::4].copy()

test_catalog = axas2_catalog.copy()
#test_catalog = axec2_catalog.copy()

print(f"Test catalog created with every 4th AXAS2 event, total {len(test_catalog)} events")
print(f"\nDate range of test catalog:")
print(f"Start: {test_catalog['datetime'].min()}")
print(f"End: {test_catalog['datetime'].max()}")

In [ ]:
# First, reformat the datetime strings to add 'T' separator
test_catalog['p_time'] = test_catalog['p_time'].str.replace(' ', 'T', regex=False)
test_catalog['s_time'] = test_catalog['s_time'].str.replace(' ', 'T', regex=False)
test_catalog['datetime'] = test_catalog['datetime'].astype(str).str.replace(' ', 'T', regex=False)

# Now convert to pandas Timestamp with UTC timezone
test_catalog['p_time'] = pd.to_datetime(test_catalog['p_time'], utc=True, format='ISO8601')
test_catalog['s_time'] = pd.to_datetime(test_catalog['s_time'], utc=True, format='ISO8601')
test_catalog['datetime'] = pd.to_datetime(test_catalog['datetime'], utc=True, format='ISO8601')

# Convert to UTCDateTime
test_catalog['p_time'] = test_catalog['p_time'].apply(lambda x: UTCDateTime(x))
test_catalog['s_time'] = test_catalog['s_time'].apply(lambda x: UTCDateTime(x))
test_catalog['datetime'] = test_catalog['datetime'].apply(lambda x: UTCDateTime(x))

print("Successfully converted to UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")

In [ ]:
catalog = test_catalog.copy()

## 3. Extended Time Window Creation

This step creates extended time windows for waveform retrieval. This is critical for proper analysis and must happen early in the workflow, before any quality control that depends on waveform data.

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols].head())

In [ ]:
# Remove leading 'OO' from station names
extended_catalog['station'] = extended_catalog['station'].str.replace('OO', '', regex=False)

In [ ]:
display(extended_catalog)

## 4. Waveform Data Retrieval

This section retrieves seismic waveform data using the extended time windows. We'll load the existing trace data and organize it for processing.

In [ ]:
test_catalog = extended_catalog

In [ ]:
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

test_catalog['mag'] = 0.0

In [ ]:
test_catalog

In [ ]:
# Retrieve waveforms for all events in the test catalog using get_all_traces function
#print("Retrieving waveforms for all events in the test catalog...")
#waveforms = get_station_traces_batch(test_catalog, 'axial_nonlinloc_april_20_28', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Load waveforms from mseed file with obspy
waveforms_file = 'axial_nonlinloc_april_20_28.mseed'
waveforms = obspy.read(waveforms_file)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if len(st) > 3:
            print(f"Event {eid}: Found {len(st)} traces (expected 3)")
            
            # Create a new stream with unique traces based on channel code
            unique_traces = {}
            for tr in st:
                channel = tr.stats.channel
                # Keep the first occurrence of each channel
                if channel not in unique_traces:
                    unique_traces[channel] = tr
            
            # Replace the stream with deduplicated traces
            organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
            print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Remove events that do not have exactly 3 traces
print("\nRemoving events that do not have exactly 3 traces...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if len(organized_waveforms[eid]['traces']) != 3:
        events_to_remove.append(eid)

    # also remove events with any trace that has zero length (indicating a retrieval issue) or empty traces
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

for eid in events_to_remove:
    del organized_waveforms[eid]

In [ ]:
len(organized_waveforms)

## 5. Quality Control Pipeline

This section implements comprehensive quality control measures including P-wave rectilinearity analysis, signal-to-noise ratio calculations, and incidence angle filtering.

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
    'min_magnitude': 0.0,      # Minimum event magnitude
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
#organized_waveforms.pop(events_to_remove[0], None)
#print(f"Removed {len(events_to_remove)} events with inconsistent trace lengths")
#print(f"Remaining events after QC: {len(organized_waveforms)}")

In [ ]:
# Check if any traces are length zero, and if so mark those events for removal
print("Checking for traces with zero length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

In [ ]:
# Redefine calculate_snr_for_organized_waveforms to skip events that return empty spec sequence, then remove them from organized_waveforms

def calculate_snr_for_organized_waveforms(organized_waveforms):
    """
    Calculate SNR for all events in organized_waveforms and add to the dataset.
    
    All required metadata (S/P arrival times, datetime) is already in organized_waveforms,
    so no external catalog lookup is needed.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing event data, traces, and metadata
        
    Returns:
    --------
    dict
        Updated organized_waveforms with SNR values added to each event
    """
    
    print(f"Calculating SNR for {len(organized_waveforms)} events in organized_waveforms...")
    
    success_count = 0
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'='*60}")
        print(f"Processing event {event_id}...")
        print(f"{'='*60}")
        
        # Get traces for this event
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  No traces found for event {event_id}")
            event_data['snr_e'] = np.nan
            event_data['snr_n'] = np.nan
            event_data['snr_horizontal'] = np.nan
            continue
        
        print(f"  Found {len(event_traces)} traces for this event")
        
        # Calculate SNR for horizontal components using event_data directly

        try:
            print("\n  Calculating SNR for E component:")
            snr_horizontal = compute_snr_for_event_baillard(event_traces.copy(), event_data)
    
        except Exception as e:
            print(f"  Error computing SNR for event {event_id}: {e}")
            snr_horizontal = np.nan

        event_data['snr_horizontal'] = snr_horizontal
        
        # Print summary
        print(f"\n  Final SNR Results:")

        if not np.isnan(snr_horizontal):
            print(f"    Horizontal average: {snr_horizontal:.2f}")
        else:
            print("    Horizontal average: N/A")

        if not np.isnan(snr_horizontal):
            success_count += 1
    
    print(f"\n{'='*60}")
    print("SNR Calculation Complete")
    print(f"{'='*60}")
    print(f"Events with valid SNR: {success_count}/{len(organized_waveforms)}")
    
    # Calculate statistics
    snr_values = [data.get('snr_horizontal', np.nan) for data in organized_waveforms.values()]
    valid_snr = [v for v in snr_values if not np.isnan(v)]
    
    if valid_snr:
        print(f"SNR range: {min(valid_snr):.2f} to {max(valid_snr):.2f}")
        print(f"Mean SNR: {np.mean(valid_snr):.2f}")
        print(f"Median SNR: {np.median(valid_snr):.2f}")
    
    else:
        # Remove events that have NaN SNR values from organized_waveforms
        print("No valid SNR values found, removing events with NaN SNR from organized_waveforms...")
        events_to_remove = [eid for eid, data in organized_waveforms.items() if np.isnan(data.get('snr_horizontal', np.nan))]
        for eid in events_to_remove:
            del organized_waveforms[eid]
        print(f"Removed {len(events_to_remove)} events with NaN SNR values")
        print(f"Remaining events after SNR QC: {len(organized_waveforms)}")
    return organized_waveforms

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Create a dataframe with quality control metrics for the test catalog events
qc_metrics_df = pd.DataFrame({
    'event_id': list(organized_waveforms.keys()),
    's_time': [organized_waveforms[eid]['s_arrival_time'] for eid in organized_waveforms.keys()],
    'station': [organized_waveforms[eid]['station'] for eid in organized_waveforms.keys()],
    'back_azimuth': [organized_waveforms[eid]['back_azimuth'] for eid in organized_waveforms.keys()],
    'snr_horizontal': [organized_waveforms[eid]['snr_horizontal'] for eid in organized_waveforms.keys()],
    'incidence': [organized_waveforms[eid]['incidence_eigenvalue_jurkevics'] for eid in organized_waveforms.keys()],
    'rectilinearity': [organized_waveforms[eid]['rectilinearity_jurkevics'] for eid in organized_waveforms.keys()],
    'latitude': [organized_waveforms[eid]['latitude'] for eid in organized_waveforms.keys()],
    'longitude': [organized_waveforms[eid]['longitude'] for eid in organized_waveforms.keys()],
    'depth': [organized_waveforms[eid]['depth'] for eid in organized_waveforms.keys()],
    'origin_time' : [organized_waveforms[eid]['origin_time'] for eid in organized_waveforms.keys()]
})

print(f"Quality control metrics dataframe created with {len(qc_metrics_df)} events")
display(qc_metrics_df)

In [ ]:
qc_metrics_df['s_time'] = qc_metrics_df['origin_time'] + qc_metrics_df['s_time']

In [ ]:
display(qc_metrics_df)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

## 6. Shear-Wave Splitting Analysis

This section implements the core shear-wave splitting analysis using SWSPy with dynamic parameter estimation and comprehensive quality assessment.

In [ ]:
def perform_splitting_on_organized_waveforms(organized_waveforms, use_dynamic_params=False, mode='swspy', plot_results=False):
    """
    Perform shear-wave splitting analysis on all events in organized_waveforms.
    
    This function assumes organized_waveforms has been filtered by apply_quality_control()
    and contains only events that pass QC thresholds. It extracts horizontal components
    and performs splitting analysis using the back_azimuth for coordinate rotation.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing QC-filtered traces and metadata
        Must have: traces, back_azimuth, station, and all other event metadata

    mode : str
        Splitting analysis method to use: 'swspy' or 'baillard' (default: 'swspy')
        
    Returns:
    --------
    dict
        Dictionary with event IDs as keys, each containing:
        {
            event_id: {
                'result': result_dict,  # Splitting parameters and metadata
                'splitting_obj': splitting_obj  # SWSPy splitting object
            },
            ...
        }
    """
    
    print(f"\n{'='*60}")
    print("Performing Shear-Wave Splitting Analysis")
    print(f"{'='*60}")
    print(f"Processing {len(organized_waveforms)} QC-filtered events...")
    
    event_results = {}
    
    # Track statistics
    stats = {
        'total_events': len(organized_waveforms),
        'missing_components': 0,
        'missing_back_azimuth': 0,
        'splitting_errors': 0,
        'successful_splits': 0
    }
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'─'*60}")
        print(f"Event {event_id}")
        print(f"{'─'*60}")
        
        # Get traces
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  ✗ No traces found")
            stats['missing_components'] += 1
            continue
        
        # Convert to stream if needed
        if isinstance(event_traces, list):
            event_stream = obspy.Stream(event_traces)
        else:
            event_stream = event_traces
        
        # Find N and E components
        trace_n = None
        trace_e = None
        trace_z = None
        
        for tr in event_stream:
            component = tr.stats.channel[-1].upper()
            if component in ['N', '1']:
                trace_n = tr
            elif component in ['E', '2']:
                trace_e = tr
            elif component == 'Z':
                trace_z = tr
        
        # Check if we have horizontal components
        if trace_n is None or trace_e is None:
            print(f"  ✗ Missing horizontal components (N: {trace_n is not None}, E: {trace_e is not None})")
            stats['missing_components'] += 1
            continue
        
        print(f"  ✓ Found horizontal components")
        
        # Check for back-azimuth
        back_azimuth = event_data.get('back_azimuth')
        if back_azimuth is None or np.isnan(back_azimuth):
            print(f"  ✗ Missing back-azimuth")
            stats['missing_back_azimuth'] += 1
            continue
        
        print(f"  ✓ Back-azimuth: {back_azimuth:.2f}°")
        
        # Get station and other metadata
        station_name = event_data.get('station', 'UNKNOWN')
        magnitude = event_data.get('magnitude', np.nan)
        snr_horizontal = event_data.get('snr_horizontal', np.nan)
        rectilinearity = event_data.get('rectilinearity_jurkevics', np.nan)
        incidence = event_data.get('incidence_eigenvalue_jurkevics', np.nan)
        
        print(f"  Station: {station_name}")
        print(f"  Magnitude: {magnitude:.1f}")
        print(f"  SNR: {snr_horizontal:.2f}")
        print(f"  Rectilinearity: {rectilinearity:.3f}")
        print(f"  Incidence: {incidence:.1f}°")
        
        # Perform splitting analysis using Baillard method
        print(f"\n  → Running Baillard splitting analysis...")
        
        try:
           
            if mode=='baillard':
                # Call Baillard splitting function
                splitting_result = perform_splitting_analysis_baillard(
                    event_data,
                    s_window=[0.02,0.3],
                    min_lag=0,
                    max_lag=60,
                    Nlags=60,
                    Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results
                )
                
                # Baillard returns dict only (no splitting_obj like SWSPy)
                splitting_obj = None

            # Hemmett-adapted SWSPy-like implementation that draws on MFAST
            elif mode=='swspy':
                # Call SWSPy-like splitting function
                splitting_result, splitting_obj = perform_splitting_analysis(
                    event_data, use_dynamic_params=use_dynamic_params, plot_results=plot_results
                )

            elif mode=='teanby_baillard':
                # Call Teanby clustering with Baillard method
                splitting_result = tbc.teanby_clustering_analysis(
                    event_data,
                    T_beg_1=0.02, T_end_0=0.3,
                    dT_beg=0.005, dT_end=0.005,
                    N_beg=10, N_end=10,
                    min_lag=0, max_lag=60,
                    Nlags=60, Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results,
                    output_dir=None,
                    N_c_min=5
                )
                splitting_obj = None

            if splitting_result.get('success', False):
                # Add additional metadata to result
                splitting_result['event_id'] = event_id
                splitting_result['back_azimuth'] = back_azimuth
                splitting_result['snr_horizontal'] = snr_horizontal
                splitting_result['rectilinearity_jurkevics'] = rectilinearity
                splitting_result['incidence_eigenvalue_jurkevics'] = incidence
                splitting_result['event_lat'] = event_data.get('latitude')
                splitting_result['event_lon'] = event_data.get('longitude')
                splitting_result['event_depth'] = event_data.get('depth')
                splitting_result['event_datetime'] = event_data.get('datetime')
                
                # Store both result dict and splitting object for later use
                event_results[event_id] = {
                    'result': splitting_result,
                    'splitting_obj': splitting_obj
                }
                stats['successful_splits'] += 1
                
                print(f"  ✓ SUCCESS!")
                if not np.isnan(splitting_result['phi']):
                    print(f"    Fast axis (φ): {splitting_result['phi']:.1f}°")
                else:
                    print(f"    Fast axis (φ): N/A")
                    
                if not np.isnan(splitting_result['dt']):
                    print(f"    Delay time (δt): {splitting_result['dt']:.3f}s")
                else:
                    print(f"    Delay time (δt): N/A")
                    
                if not np.isnan(splitting_result.get('phi_error', np.nan)):
                    print(f"    φ error: ±{splitting_result['phi_error']:.1f}°")
                if not np.isnan(splitting_result.get('dt_error', np.nan)):
                    print(f"    δt error: ±{splitting_result['dt_error']:.3f}s")
                if 'dominant_period' in splitting_result:
                    print(f"    Dominant period: {splitting_result['dominant_period']:.3f}s")
            else:
                error_msg = splitting_result.get('error', 'Unknown error')
                print(f"  ✗ FAILED: {error_msg}")
                if 'traceback' in splitting_result:
                    print(f"  Traceback:\n{splitting_result['traceback']}")
                stats['splitting_errors'] += 1
                
        except Exception as e:
            import traceback
            print(f"  ✗ ERROR during splitting: {e}")
            print(f"  Traceback:\n{traceback.format_exc()}")
            stats['splitting_errors'] += 1
            continue
    
    # Print summary
    print(f"\n{'='*60}")
    print("Splitting Analysis Summary")
    print(f"{'='*60}")
    print(f"Total events processed: {stats['total_events']}")
    print(f"Successful splits: {stats['successful_splits']} ({100*stats['successful_splits']/stats['total_events']:.1f}%)")
    print(f"\nFailure breakdown:")
    print(f"  Missing components: {stats['missing_components']}")
    print(f"  Missing back-azimuth: {stats['missing_back_azimuth']}")
    print(f"  Splitting errors: {stats['splitting_errors']}")
    
    # Calculate splitting parameter statistics if we have results
    if event_results:

        if mode!='swspy':
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi_rad'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi_rad'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        else:
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        if phi_values and dt_values:
            print(f"\n{'─'*60}")
            print("Splitting Parameter Statistics")
            print(f"{'─'*60}")
            print(f"Fast axis direction (φ):")
            print(f"  Mean: {np.mean(phi_values):.1f}° ± {np.std(phi_values):.1f}°")
            print(f"  Range: {np.min(phi_values):.1f}° to {np.max(phi_values):.1f}°")
            print(f"  Median: {np.median(phi_values):.1f}°")
            
            print(f"\nDelay time (δt):")
            print(f"  Mean: {np.mean(dt_values):.3f} ± {np.std(dt_values):.3f}s")
            print(f"  Range: {np.min(dt_values):.3f}s to {np.max(dt_values):.3f}s")
            print(f"  Median: {np.median(dt_values):.3f}s")
            
            # Add to stats
            stats['phi_mean'] = float(np.mean(phi_values))
            stats['phi_std'] = float(np.std(phi_values))
            stats['dt_mean'] = float(np.mean(dt_values))
            stats['dt_std'] = float(np.std(dt_values))
        else:
            print(f"\n  Note: No valid splitting parameters for statistics")
    
    # Return the event_results dictionary directly
    # Each entry contains both 'result' and 'splitting_obj'
    return event_results

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, use_dynamic_params=False, mode='baillard', plot_results=False)

In [ ]:
# Save results_baillard dictionary to CSV file for later analysis
results_df = pd.DataFrame.from_dict(results_baillard, orient='index')
results_df.to_csv('results_baillard_nonlinloc_april_20_28.csv')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy = perform_splitting_on_organized_waveforms(passing_waveforms, use_dynamic_params=False, mode='swspy', plot_results=False)

In [ ]:
# Save results_swspy dictionary to CSV file for later analysis
results_swspy_df = pd.DataFrame.from_dict(results_swspy, orient='index')
results_swspy_df.to_csv('results_swspy_nonlinloc_april_20_28.csv')

In [ ]:
# Save results_swspy dictionary to CSV file for later analysis - SWSPy out of the box, setting appropriate parameters for this dataset
#results_swspy_df = pd.DataFrame.from_dict(results_swspy, orient='index')
#results_swspy_df.to_csv('results_swspy_base_nonlinloc_april_20_28.csv')

In [ ]:
# Load results from CSV files
results_baillard_df = pd.read_csv('results_baillard_nonlinloc_april_20_28.csv', index_col=0)
results_swspy_df = pd.read_csv('results_swspy_nonlinloc_april_20_28.csv', index_col=0)

In [ ]:
# Convert df to dictionary for plotting
results_baillard_dict = results_baillard_df.to_dict(orient='index')
results_swspy_dict = results_swspy_df.to_dict(orient='index')

In [ ]:
# Keep only events in results_baillard that are also in results_swspy for direct comparison
common_event_ids = set(results_baillard.keys()) & set(results_swspy.keys())
results_baillard_common = {eid: results_baillard[eid] for eid in common_event_ids}
results_swspy_common = {eid: results_swspy[eid] for eid in common_event_ids}

In [ ]:
def plot_fast_direction_rose(results_dict, title="Fast Direction Distribution", 
                              nbins=18, figsize=(8, 8), color='steelblue',
                              edgecolor='black', linewidth=0.5):
    """
    Create a polar rose plot (histogram) of fast directions from splitting results.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 18 = 10° bins for ±90°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']  # in degrees
        # Convert to radians
        phi_rad = np.deg2rad(phi)
        fast_directions.append(phi_rad)
    
    fast_directions = np.array(fast_directions)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (-pi/2 to pi/2 for -90° to +90°)
    bins = np.linspace(-np.pi/2, np.pi/2, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    # Set angular limits (-90° to +90°)
    ax.set_thetamin(-90)
    ax.set_thetamax(90)
    
    # Set radial ticks
    ax.set_rlabel_position(0)
    
    # Add degree labels
    tick_labels = ['-90°', '-60°', '-30°', '0°', '30°', '60°', '90°']
    tick_positions = np.deg2rad([-90, -60, -30, 0, 30, 60, 90])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Add title with statistics
    n_measurements = len(fast_directions)
    # Calculate circular mean for ±90° range
    mean_direction = np.rad2deg(np.arctan2(np.sin(fast_directions).sum(), 
                                           np.cos(fast_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax


In [ ]:
# Convert results string indices to integers from string indices
results_baillard_int = {int(eid): result for eid, result in results_baillard.items()}
results_swspy_int = {int(eid): result for eid, result in results_swspy.items()}

In [ ]:
if 'results_swspy' in locals() and results_swspy:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy_converted,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy - Baillard Phi Convention",
        nbins=18,  # 10° bins
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

In [ ]:
# Create the rose plot
fig, ax = plot_fast_direction_rose(
    results_baillard_common,
    title=f"Fast Direction Rose Plot - Station AXAS2, Baillard",
    nbins=18,  # 10° bins
    color='steelblue',
    figsize=(10, 10)
)
plt.show()

In [ ]:
def plot_splitting_timeseries_smooth(results_dict, qc_metrics_df, station='AXAS2', 
                                     figsize=(14, 8), x_width_days=5, x_overlap=0.95,
                                     y_width_phi=5, y_width_dt=2, y_overlap=0.95,
                                     sigma=2.0, sampling_rate=200.0):
    """
    Create smoothed 2D histogram time-series plots inspired by Baillard's approach.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    qc_metrics_df : pd.DataFrame
        DataFrame with event metadata including origin_time
    station : str
        Station name for title
    figsize : tuple
        Figure size (width, height)
    x_width_days : float
        Width of moving time window in days
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width_phi : float
        Bin width for phi in radians
    y_width_dt : int
        Bin width for dt in samples (1 sample = 1/sampling_rate seconds)
    y_overlap : float
        Overlap for smoothing in y-direction
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.dates import DateFormatter
    import matplotlib.dates as mdates
    from scipy.ndimage import gaussian_filter
    
    # Extract data from results
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = qc_metrics_df.loc[qc_metrics_df['event_id'] == event_id, 'origin_time'].values[0]
            phi = result['result']['phi']
            dt = result['result']['dt']
            
            # Convert phi from degrees to radians and normalize to -pi/2 to +pi/2
            phi_rad = np.deg2rad(phi)
            phi_rad_norm = ((phi_rad + np.pi/2) % np.pi) - np.pi/2
            
            # Convert dt from seconds to samples
            dt_samples = dt * sampling_rate
            
            data_list.append({
                'time': pd.to_datetime(str(origin_time)),
                'phi_rad': phi_rad_norm,
                'phi_deg': phi,
                'dt_samples': dt_samples,
                'dt_seconds': dt
            })
    
    df = pd.DataFrame(data_list).sort_values('time')
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Convert datetime to matplotlib date numbers
    time_nums = mdates.date2num(df['time'])
    
    # === PHI PLOT (in radians) ===
    # Create bins
    time_range = (time_nums.min(), time_nums.max())
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_days * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))  # Reasonable limits
    
    # Phi bins from -pi/2 to +pi/2 radians (-1.57 to +1.57)
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    n_phi_bins = len(phi_bins) - 1
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_nums, df['phi_rad'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin) - "norm_y=True" in Baillard's code
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    
    # Mask zeros
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot with imshow for smooth appearance
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',  # Smooth interpolation
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis with radians
    ax_phi.set_ylabel('Fast Direction φ (rad)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')

      # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_phi.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Set y-ticks in radians
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    ax_phi.set_yticks(phi_ticks_rad)
    
    # Add moving average
    window_size = max(5, len(df) // 10)
    if len(df) >= window_size:
        df['phi_rad_ma'] = df['phi_rad'].rolling(window=window_size, center=True).mean()
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_phi.plot(df['time'], df['phi_rad_ma'], 'black', linewidth=2,
                   label=f'{window_size}-event moving avg', alpha=0.8)
        ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
                     edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = df['dt_samples'].quantile(0.98)
    # Create bins in samples
    dt_bins = np.arange(0, min(40, dt_max_samples) + y_width_dt, y_width_dt)  # 40 samples = 0.2 sec at 200 Hz
    n_dt_bins = len(dt_bins) - 1
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_nums, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    
    # Mask zeros
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot with imshow
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)

    # Add axvline dashed white line at time = 2015-04-24 05:00:00
    event_time_line = pd.to_datetime('2015-04-24 05:00:00')
    ax_dt.axvline(mdates.date2num(event_time_line), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Format dt axis with samples
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, 25)  # Show up to 25 samples (0.125 sec at 200 Hz)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add secondary y-axis for seconds
    #ax_dt_sec = ax_dt.secondary_yaxis('right', functions=(
    #    lambda x: x / sampling_rate,  # samples to seconds
    #    lambda x: x * sampling_rate   # seconds to samples
    #))
    #ax_dt_sec.set_ylabel('δt (s)', fontsize=10)
    
    # Add moving average for dt
    if len(df) >= window_size:
        df['dt_samples_ma'] = df['dt_samples'].rolling(window=window_size, center=True).mean()
        ax_dt.plot(df['time'], df['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_dt.plot(df['time'], df['dt_samples_ma'], 'black', linewidth=2,
                  label=f'{window_size}-event moving avg', alpha=0.8)
        ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
                    edgecolor='white', framealpha=0.7)
    
    # Format x-axis with dates
    date_formatter = DateFormatter('%Y-%m-%d')
    ax_dt.xaxis.set_major_formatter(date_formatter)
    
    # Auto-adjust date locator
    days_span = (df['time'].max() - df['time'].min()).days
    if days_span > 60:
        ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    elif days_span > 14:
        ax_dt.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    else:
        ax_dt.xaxis.set_major_locator(mdates.DayLocator())
    
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics (in both degrees and radians)
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    title = (f"Splitting Parameter Time Series - Station {station}\n"
             f"N = {len(df)} events | "
             f"φ: {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}° "
             f"({np.deg2rad(phi_mean_deg):.2f} ± {np.deg2rad(phi_std_deg):.2f} rad) | "
             f"δt: {dt_mean_sec:.3f} ± {dt_std_sec:.3f} s "
             f"({dt_mean_sec*sampling_rate:.1f} ± {dt_std_sec*sampling_rate:.1f} samples)")
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df

In [ ]:
# Make a new list of phi results from SWSPy, converted to Baillard convention for direct comparison
# phi_baillard = ((90 + phi + 90) % 180) - 90
def convert_phi_baillard(phi_deg):
    return -1 * (((90 + phi_deg + 90) % 180) - 90)

# Create a new results dictionary with converted phi values for SWSPy results
results_swspy_converted = {}
for event_id, result in results_swspy.items():
    phi_swspy = result['result']['phi']
    phi_converted = convert_phi_baillard(phi_swspy)
    
    # Create a new result dict with converted phi but same dt and metadata
    converted_result = result.copy()
    converted_result['result'] = converted_result['result'].copy()
    converted_result['result']['phi'] = phi_converted  # Update phi to Baillard convention
    
    results_swspy_converted[event_id] = converted_result

In [ ]:
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy_converted,
    qc_metrics_df,
    station='AXAS2, SWSPy - Baillard Phi Convention',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
results_swspy_converted = results_swspy.copy()

In [ ]:
# Print first few phi results from both methods
print("=" * 80)
print("First 10 Phi Results Comparison")
print("=" * 80)
print(f"{'Event ID':<12} {'SWSPy→Baillard (°)':<20} {'Baillard (°)':<15} {'Difference (°)':<15}")
print("-" * 80)

# Get common event IDs
common_ids = sorted(list(set(results_swspy_converted.keys()) & set(results_baillard.keys())))[:10]

for eid in common_ids:
    phi_swspy = results_swspy_converted[eid]['result']['phi']
    phi_baillard = results_baillard[eid]['result']['phi']
    diff = phi_swspy - phi_baillard
    print(f"{eid:<12} {phi_swspy:>18.2f} {phi_baillard:>14.2f} {diff:>14.2f}")

print("\n")

# Calculate residuals for all common events
common_event_ids = set(results_swspy_converted.keys()) & set(results_baillard.keys())
print(f"Total common events: {len(common_event_ids)}\n")

phi_swspy_list = []
phi_baillard_list = []
phi_residuals = []
dt_swspy_list = []
dt_baillard_list = []
dt_residuals = []

for eid in common_event_ids:
    phi_s = results_swspy_converted[eid]['result']['phi']
    phi_b = results_baillard[eid]['result']['phi']
    dt_s = results_swspy[eid]['result']['dt']
    dt_b = results_baillard[eid]['result']['dt']
    
    phi_swspy_list.append(phi_s)
    phi_baillard_list.append(phi_b)
    phi_residuals.append(phi_s - phi_b)
    
    dt_swspy_list.append(dt_s)
    dt_baillard_list.append(dt_b)
    dt_residuals.append(dt_s - dt_b)

# Convert to numpy arrays
phi_swspy_arr = np.array(phi_swspy_list)
phi_baillard_arr = np.array(phi_baillard_list)
phi_residuals_arr = np.array(phi_residuals)
dt_swspy_arr = np.array(dt_swspy_list)
dt_baillard_arr = np.array(dt_baillard_list)
dt_residuals_arr = np.array(dt_residuals)

# Print statistics
print("Phi (Fast Direction) Statistics:")
print(f"  Mean residual: {np.mean(phi_residuals_arr):.2f}°")
print(f"  Std residual: {np.std(phi_residuals_arr):.2f}°")
print(f"  Min residual: {np.min(phi_residuals_arr):.2f}°")
print(f"  Max residual: {np.max(phi_residuals_arr):.2f}°")
print(f"  Median residual: {np.median(phi_residuals_arr):.2f}°")

print("\nDelay Time (dt) Statistics:")
print(f"  Mean residual: {np.mean(dt_residuals_arr):.4f}s")
print(f"  Std residual: {np.std(dt_residuals_arr):.4f}s")
print(f"  Min residual: {np.min(dt_residuals_arr):.4f}s")
print(f"  Max residual: {np.max(dt_residuals_arr):.4f}s")
print(f"  Median residual: {np.median(dt_residuals_arr):.4f}s")

# Create comprehensive residual plots
fig = plt.figure(figsize=(16, 12))
gs = plt.GridSpec(3, 2, hspace=0.3, wspace=0.3, top=0.95, bottom=0.06, left=0.08, right=0.96)

# 1. Phi scatter plot (SWSPy vs Baillard)
ax1 = plt.subplot(gs[0, 0])
ax1.scatter(phi_baillard_arr, phi_swspy_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
ax1.plot([-90, 90], [-90, 90], 'r--', linewidth=2, label='1:1 line')
ax1.set_xlabel('Baillard φ (°)', fontsize=12, fontweight='bold')
ax1.set_ylabel('SWSPy φ (°, Baillard convention)', fontsize=12, fontweight='bold')
ax1.set_title('Fast Direction Comparison', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_xlim(-90, 90)
ax1.set_ylim(-90, 90)

# Add correlation coefficient
corr_phi = np.corrcoef(phi_baillard_arr, phi_swspy_arr)[0, 1]
ax1.text(0.05, 0.95, f'r = {corr_phi:.3f}', transform=ax1.transAxes,
         fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. Phi residuals histogram
ax2 = plt.subplot(gs[0, 1])
ax2.hist(phi_residuals_arr, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero residual')
ax2.axvline(np.mean(phi_residuals_arr), color='orange', linestyle='-', linewidth=2, 
            label=f'Mean: {np.mean(phi_residuals_arr):.2f}°')
ax2.set_xlabel('Residual (SWSPy - Baillard) (°)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count', fontsize=12, fontweight='bold')
ax2.set_title('Fast Direction Residuals Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend()

# 3. dt scatter plot (SWSPy vs Baillard)
ax3 = plt.subplot(gs[1, 0])
ax3.scatter(dt_baillard_arr, dt_swspy_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5, color='coral')
max_dt = max(dt_baillard_arr.max(), dt_swspy_arr.max())
ax3.plot([0, max_dt], [0, max_dt], 'r--', linewidth=2, label='1:1 line')
ax3.set_xlabel('Baillard δt (s)', fontsize=12, fontweight='bold')
ax3.set_ylabel('SWSPy δt (s)', fontsize=12, fontweight='bold')
ax3.set_title('Delay Time Comparison', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Add correlation coefficient
corr_dt = np.corrcoef(dt_baillard_arr, dt_swspy_arr)[0, 1]
ax3.text(0.05, 0.95, f'r = {corr_dt:.3f}', transform=ax3.transAxes,
         fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 4. dt residuals histogram
ax4 = plt.subplot(gs[1, 1])
ax4.hist(dt_residuals_arr, bins=30, edgecolor='black', alpha=0.7, color='coral')
ax4.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero residual')
ax4.axvline(np.mean(dt_residuals_arr), color='orange', linestyle='-', linewidth=2,
            label=f'Mean: {np.mean(dt_residuals_arr):.4f}s')
ax4.set_xlabel('Residual (SWSPy - Baillard) (s)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Count', fontsize=12, fontweight='bold')
ax4.set_title('Delay Time Residuals Distribution', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
ax4.legend()

# 5. Phi residuals vs Baillard phi (check for bias)
ax5 = plt.subplot(gs[2, 0])
ax5.scatter(phi_baillard_arr, phi_residuals_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
ax5.axhline(0, color='red', linestyle='--', linewidth=2)
ax5.axhline(np.mean(phi_residuals_arr), color='orange', linestyle='-', linewidth=2)
ax5.set_xlabel('Baillard φ (°)', fontsize=12, fontweight='bold')
ax5.set_ylabel('Residual (SWSPy - Baillard) (°)', fontsize=12, fontweight='bold')
ax5.set_title('Phi Residuals vs Baillard Values', fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6. dt residuals vs Baillard dt (check for bias)
ax6 = plt.subplot(gs[2, 1])
ax6.scatter(dt_baillard_arr, dt_residuals_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5, color='coral')
ax6.axhline(0, color='red', linestyle='--', linewidth=2)
ax6.axhline(np.mean(dt_residuals_arr), color='orange', linestyle='-', linewidth=2)
ax6.set_xlabel('Baillard δt (s)', fontsize=12, fontweight='bold')
ax6.set_ylabel('Residual (SWSPy - Baillard) (s)', fontsize=12, fontweight='bold')
ax6.set_title('dt Residuals vs Baillard Values', fontsize=13, fontweight='bold')
ax6.grid(True, alpha=0.3)

fig.suptitle(f'SWSPy vs Baillard Method Comparison (N = {len(common_event_ids)} events)',
             fontsize=16, fontweight='bold')

plt.show()

# Additional: Bland-Altman style plot for both parameters
fig2, (ax_phi_ba, ax_dt_ba) = plt.subplots(1, 2, figsize=(14, 5))

# Phi Bland-Altman
phi_mean = (phi_swspy_arr + phi_baillard_arr) / 2
ax_phi_ba.scatter(phi_mean, phi_residuals_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
ax_phi_ba.axhline(np.mean(phi_residuals_arr), color='blue', linestyle='-', linewidth=2, label='Mean difference')
ax_phi_ba.axhline(np.mean(phi_residuals_arr) + 1.96*np.std(phi_residuals_arr), 
                  color='red', linestyle='--', linewidth=2, label='±1.96 SD')
ax_phi_ba.axhline(np.mean(phi_residuals_arr) - 1.96*np.std(phi_residuals_arr), 
                  color='red', linestyle='--', linewidth=2)
ax_phi_ba.set_xlabel('Mean of SWSPy and Baillard φ (°)', fontsize=12, fontweight='bold')
ax_phi_ba.set_ylabel('Difference (SWSPy - Baillard) (°)', fontsize=12, fontweight='bold')
ax_phi_ba.set_title('Bland-Altman Plot: Fast Direction', fontsize=13, fontweight='bold')
ax_phi_ba.grid(True, alpha=0.3)
ax_phi_ba.legend()

# dt Bland-Altman
dt_mean = (dt_swspy_arr + dt_baillard_arr) / 2
ax_dt_ba.scatter(dt_mean, dt_residuals_arr, alpha=0.5, s=50, edgecolors='black', linewidth=0.5, color='coral')
ax_dt_ba.axhline(np.mean(dt_residuals_arr), color='blue', linestyle='-', linewidth=2, label='Mean difference')
ax_dt_ba.axhline(np.mean(dt_residuals_arr) + 1.96*np.std(dt_residuals_arr), 
                 color='red', linestyle='--', linewidth=2, label='±1.96 SD')
ax_dt_ba.axhline(np.mean(dt_residuals_arr) - 1.96*np.std(dt_residuals_arr), 
                 color='red', linestyle='--', linewidth=2)
ax_dt_ba.set_xlabel('Mean of SWSPy and Baillard δt (s)', fontsize=12, fontweight='bold')
ax_dt_ba.set_ylabel('Difference (SWSPy - Baillard) (s)', fontsize=12, fontweight='bold')
ax_dt_ba.set_title('Bland-Altman Plot: Delay Time', fontsize=13, fontweight='bold')
ax_dt_ba.grid(True, alpha=0.3)
ax_dt_ba.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes, df = plot_splitting_timeseries_smooth(
    results_baillard,
    qc_metrics_df,
    station='AXAS2, Baillard',
    x_overlap= 0.9,
    y_width_phi=4.5,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)  # Increase for more smoothing
plt.show()

In [ ]:
def plot_splitting_timeseries_baillard_style(
        results_dict,
        qc_metrics_df,
        station='AXAS2',
        figsize=(14, 8),
        x_width_days=5,
        x_overlap=0.97,
        y_width_phi=5,
        y_width_dt=1,
        sampling_rate=200.0,
        sigma_time=2.0,
        sigma_y=0.8):

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from matplotlib.dates import DateFormatter
    from scipy.ndimage import gaussian_filter

    # --------------------------------------------------
    # Extract data
    # --------------------------------------------------
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = qc_metrics_df.loc[
                qc_metrics_df['event_id'] == event_id,
                'origin_time'
            ].values[0]

            phi = result['result']['phi']
            dt = result['result']['dt']

            phi_rad = np.deg2rad(phi)
            phi_rad = ((phi_rad + np.pi/2) % np.pi) - np.pi/2
            dt_samples = dt * sampling_rate

            data_list.append({
                'time': pd.to_datetime(str(origin_time)),
                'phi_rad': phi_rad,
                'dt_samples': dt_samples
            })

    df = pd.DataFrame(data_list).sort_values('time')
    if len(df) == 0:
        print("No data to plot.")
        return

    time_nums = mdates.date2num(df['time'])

    # --------------------------------------------------
    # Sliding window density function
    # --------------------------------------------------
    def sliding_density(values, y_bins):

        t_min = time_nums.min()
        t_max = time_nums.max()

        step = x_width_days * (1 - x_overlap)
        centers = np.arange(
            t_min + x_width_days/2,
            t_max - x_width_days/2,
            step
        )

        H = np.zeros((len(centers), len(y_bins) - 1))

        for i, c in enumerate(centers):
            mask = (
                (time_nums >= c - x_width_days/2) &
                (time_nums <= c + x_width_days/2)
            )

            if np.sum(mask) > 0:
                hist, _ = np.histogram(values[mask], bins=y_bins)
                H[i, :] = hist

        # Smooth (anisotropic)
        H = gaussian_filter(H, sigma=(sigma_time, sigma_y))

        # Column normalization AFTER smoothing
        for i in range(H.shape[0]):
            s = H[i, :].sum()
            if s > 0:
                H[i, :] /= s

        H = np.ma.masked_where(H <= 0, H)

        return H, centers

    # --------------------------------------------------
    # Create figure
    # --------------------------------------------------
    fig, (ax_phi, ax_dt) = plt.subplots(
        2, 1, figsize=figsize, sharex=True
    )

    # --------------------------------------------------
    # PHI
    # --------------------------------------------------
    phi_bins = np.arange(
        -np.pi/2,
        np.pi/2 + np.deg2rad(y_width_phi),
        np.deg2rad(y_width_phi)
    )

    H_phi, t_centers = sliding_density(
        df['phi_rad'].values,
        phi_bins
    )

    extent_phi = [
        t_centers[0], t_centers[-1],
        phi_bins[0], phi_bins[-1]
    ]

    im_phi = ax_phi.imshow(
        H_phi.T,
        origin='lower',
        aspect='auto',
        extent=extent_phi,
        cmap='magma',
        interpolation='nearest'
    )

    ax_phi.set_ylabel('Fast dir (rad)', fontsize=12)
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.grid(False)

    # eruption line
    eruption_time = pd.to_datetime('2015-04-24 05:00:00')
    ax_phi.axvline(
        mdates.date2num(eruption_time),
        color='white',
        linestyle=':',
        linewidth=2
    )

    # --------------------------------------------------
    # DT
    # --------------------------------------------------
    dt_bins = np.arange(0, 30, y_width_dt)

    H_dt, t_centers = sliding_density(
        df['dt_samples'].values,
        dt_bins
    )

    extent_dt = [
        t_centers[0], t_centers[-1],
        dt_bins[0], dt_bins[-1]
    ]

    im_dt = ax_dt.imshow(
        H_dt.T,
        origin='lower',
        aspect='auto',
        extent=extent_dt,
        cmap='magma',
        interpolation='nearest'
    )

    ax_dt.set_ylabel('Lag (samples)', fontsize=12)
    ax_dt.set_xlabel('Date', fontsize=12)
    ax_dt.set_ylim(0, 30)
    ax_dt.grid(False)

    ax_dt.axvline(
        mdates.date2num(eruption_time),
        color='white',
        linestyle=':',
        linewidth=2
    )

    # --------------------------------------------------
    # Formatting
    # --------------------------------------------------
    ax_dt.xaxis.set_major_formatter(DateFormatter('%Y-%m'))
    ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45)

    fig.suptitle(
        f'{station}',
        fontsize=16,
        fontweight='bold'
    )

    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df


In [ ]:
def plot_splitting_timeseries_smooth_before_after(results_dict, qc_metrics_df, station='AXAS2', 
                                                  figsize=(14, 8), x_width_days=5, x_overlap=0.95,
                                                  y_width_phi=5, y_width_dt=2, y_overlap=0.95,
                                                  sigma=2.0, sampling_rate=200.0,
                                                  eruption_time='2015-04-24 05:00:00'):
    """
    Create smoothed 2D histogram time-series plots with separate rolling averages before/after eruption.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    qc_metrics_df : pd.DataFrame
        DataFrame with event metadata including origin_time
    station : str
        Station name for title
    figsize : tuple
        Figure size (width, height)
    x_width_days : float
        Width of moving time window in days
    x_overlap : float
        Overlap fraction for time windows (0.95 = 95% overlap)
    y_width_phi : float
        Bin width for phi in radians
    y_width_dt : int
        Bin width for dt in samples (1 sample = 1/sampling_rate seconds)
    y_overlap : float
        Overlap for smoothing in y-direction
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    eruption_time : str
        Eruption time for splitting the rolling averages (default: '2015-04-24 05:00:00')
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.dates import DateFormatter
    import matplotlib.dates as mdates
    from scipy.ndimage import gaussian_filter
    
    # Convert eruption time to pandas Timestamp
    eruption_timestamp = pd.to_datetime(eruption_time)
    
    # Extract data from results
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = qc_metrics_df.loc[qc_metrics_df['event_id'] == event_id, 'origin_time'].values[0]
            phi = result['result']['phi']
            dt = result['result']['dt']
            
            # Convert phi from degrees to radians and normalize to -pi/2 to +pi/2
            phi_rad = np.deg2rad(phi)
            phi_rad_norm = ((phi_rad + np.pi/2) % np.pi) - np.pi/2
            
            # Convert dt from seconds to samples
            dt_samples = dt * sampling_rate
            
            data_list.append({
                'time': pd.to_datetime(str(origin_time)),
                'phi_rad': phi_rad_norm,
                'phi_deg': phi,
                'dt_samples': dt_samples,
                'dt_seconds': dt
            })
    
    df = pd.DataFrame(data_list).sort_values('time')
    
    if len(df) == 0:
        print("No data to plot")
        return
    
    # Split data into before and after eruption
    df_before = df[df['time'] < eruption_timestamp].copy()
    df_after = df[df['time'] >= eruption_timestamp].copy()
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Convert datetime to matplotlib date numbers
    time_nums = mdates.date2num(df['time'])
    
    # === PHI PLOT (in radians) ===
    # Create bins
    time_range = (time_nums.min(), time_nums.max())
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_days * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))  # Reasonable limits
    
    # Phi bins from -pi/2 to +pi/2 radians (-1.57 to +1.57)
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    n_phi_bins = len(phi_bins) - 1
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_nums, df['phi_rad'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin) - "norm_y=True" in Baillard's code
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    
    # Mask zeros
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot with imshow for smooth appearance
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',  # Smooth interpolation
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis with radians
    ax_phi.set_ylabel('Fast Direction φ (rad)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')

    # Add axvline dashed white line at eruption time
    ax_phi.axvline(mdates.date2num(eruption_timestamp), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Set y-ticks in radians
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    ax_phi.set_yticks(phi_ticks_rad)
    
    # Add separate moving averages for before and after eruption
    window_size = max(5, len(df) // 10)
    
    # Before eruption rolling average (phi)
    if len(df_before) >= 3:
        window_before = max(3, len(df_before) // 5)
        df_before['phi_rad_ma'] = df_before['phi_rad'].rolling(window=window_before, center=True, min_periods=1).mean()
        ax_phi.plot(df_before['time'], df_before['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_phi.plot(df_before['time'], df_before['phi_rad_ma'], 'blue', linewidth=2, 
                   label=f'Before: {window_before}-event moving avg', alpha=0.8)
    
    # After eruption rolling average (phi)
    if len(df_after) >= 3:
        window_after = max(3, len(df_after) // 5)
        df_after['phi_rad_ma'] = df_after['phi_rad'].rolling(window=window_after, center=True, min_periods=1).mean()
        ax_phi.plot(df_after['time'], df_after['phi_rad_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_phi.plot(df_after['time'], df_after['phi_rad_ma'], 'red', linewidth=2, 
                   label=f'After: {window_after}-event moving avg', alpha=0.8)
    
    ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
                 edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = df['dt_samples'].quantile(0.98)
    # Create bins in samples
    dt_bins = np.arange(0, min(40, dt_max_samples) + y_width_dt, y_width_dt)  # 40 samples = 0.2 sec at 200 Hz
    n_dt_bins = len(dt_bins) - 1
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_nums, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    
    # Mask zeros
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot with imshow
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)

    # Add axvline dashed white line at eruption time
    ax_dt.axvline(mdates.date2num(eruption_timestamp), color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    
    # Format dt axis with samples
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, 25)  # Show up to 25 samples (0.125 sec at 200 Hz)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add separate moving averages for before and after eruption (dt)
    # Before eruption rolling average (dt)
    if len(df_before) >= 3:
        window_before = max(3, len(df_before) // 5)
        df_before['dt_samples_ma'] = df_before['dt_samples'].rolling(window=window_before, center=True, min_periods=1).mean()
        ax_dt.plot(df_before['time'], df_before['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_dt.plot(df_before['time'], df_before['dt_samples_ma'], 'blue', linewidth=2,
                  label=f'Before: {window_before}-event moving avg', alpha=0.8)
    
    # After eruption rolling average (dt)
    if len(df_after) >= 3:
        window_after = max(3, len(df_after) // 5)
        df_after['dt_samples_ma'] = df_after['dt_samples'].rolling(window=window_after, center=True, min_periods=1).mean()
        ax_dt.plot(df_after['time'], df_after['dt_samples_ma'], 'w-', linewidth=2.5, alpha=0.9)
        ax_dt.plot(df_after['time'], df_after['dt_samples_ma'], 'red', linewidth=2,
                  label=f'After: {window_after}-event moving avg', alpha=0.8)
    
    ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
                edgecolor='white', framealpha=0.7)
    
    # Format x-axis with dates
    date_formatter = DateFormatter('%Y-%m-%d')
    ax_dt.xaxis.set_major_formatter(date_formatter)
    
    # Auto-adjust date locator
    days_span = (df['time'].max() - df['time'].min()).days
    if days_span > 60:
        ax_dt.xaxis.set_major_locator(mdates.MonthLocator())
    elif days_span > 14:
        ax_dt.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    else:
        ax_dt.xaxis.set_major_locator(mdates.DayLocator())
    
    plt.setp(ax_dt.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics (in both degrees and radians)
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    # Add before/after statistics
    stats_lines = [f"N = {len(df)} events total"]
    if len(df_before) > 0:
        stats_lines.append(f"Before: N={len(df_before)}, φ={df_before['phi_deg'].mean():.1f}°±{df_before['phi_deg'].std():.1f}°, δt={df_before['dt_seconds'].mean():.3f}±{df_before['dt_seconds'].std():.3f}s")
    if len(df_after) > 0:
        stats_lines.append(f"After: N={len(df_after)}, φ={df_after['phi_deg'].mean():.1f}°±{df_after['phi_deg'].std():.1f}°, δt={df_after['dt_seconds'].mean():.3f}±{df_after['dt_seconds'].std():.3f}s")
    
    title = (f"Splitting Parameter Time Series - Station {station}\n" + 
             " | ".join(stats_lines))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df

In [ ]:
def plot_splitting_timeseries_around_eruption(results_dict, qc_metrics_df, 
                                               eruption_time, station='AXAS2',
                                               hours_before=24, hours_after=24,
                                               figsize=(14, 8), x_width_hours=2, x_overlap=0.95,
                                               y_width_phi=5, y_width_dt=2,
                                               sigma=2.0, sampling_rate=200.0):
    """
    Create smoothed 2D histogram time-series plots focused on eruption timing.
    Time axis shows hours relative to eruption (negative = before, positive = after).
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    qc_metrics_df : pd.DataFrame
        DataFrame with event metadata including origin_time
    eruption_time : str or pd.Timestamp
        Eruption onset time (e.g., "2015-04-24 05:00:00")
    station : str
        Station name for title
    hours_before : float
        Hours before eruption to include (default: 24)
    hours_after : float
        Hours after eruption to include (default: 24)
    figsize : tuple
        Figure size (width, height)
    x_width_hours : float
        Width of moving time window in hours
    x_overlap : float
        Overlap fraction for time windows
    y_width_phi : float
        Bin width for phi in degrees
    y_width_dt : int
        Bin width for dt in samples
    sigma : float
        Gaussian smoothing parameter
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    """
    import matplotlib.gridspec as gridspec
    from scipy.ndimage import gaussian_filter
    
    # Convert eruption time to pandas Timestamp
    eruption_time = pd.to_datetime(eruption_time)
    
    # Define time window
    start_time = eruption_time - pd.Timedelta(hours=hours_before)
    end_time = eruption_time + pd.Timedelta(hours=hours_after)
    
    # Extract data from results within time window
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = pd.to_datetime(str(qc_metrics_df.loc[
                qc_metrics_df['event_id'] == event_id, 'origin_time'].values[0]))
            
            # Filter by time window
            if start_time <= origin_time <= end_time:
                phi = result['result']['phi']
                dt = result['result']['dt']
                
                # Calculate hours relative to eruption (negative = before)
                hours_from_eruption = (origin_time - eruption_time).total_seconds() / 3600.0
                
                # Convert phi from degrees to radians and normalize to -pi/2 to +pi/2
                phi_rad = np.deg2rad(phi)
                phi_rad_norm = ((phi_rad + np.pi/2) % np.pi) - np.pi/2
                
                # Convert dt from seconds to samples
                dt_samples = dt * sampling_rate
                
                data_list.append({
                    'hours_from_eruption': hours_from_eruption,
                    'phi_rad': phi_rad_norm,
                    'phi_deg': phi,
                    'dt_samples': dt_samples,
                    'dt_seconds': dt
                })
    
    df = pd.DataFrame(data_list).sort_values('hours_from_eruption')
    
    if len(df) == 0:
        print(f"No data found in time window [{start_time} to {end_time}]")
        return None, None, None
    
    print(f"Found {len(df)} events in window ({hours_before}h before to {hours_after}h after eruption)")
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.9, bottom=0.1, 
                          left=0.08, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Time range in hours
    time_vals = df['hours_from_eruption'].values
    time_range = (time_vals.min(), time_vals.max())
    
    # === PHI PLOT (in radians) ===
    # Create bins
    n_time_bins = int((time_range[1] - time_range[0]) / (x_width_hours * (1 - x_overlap)))
    n_time_bins = max(20, min(n_time_bins, 100))
    
    # Phi bins from -pi/2 to +pi/2 radians
    phi_bins = np.arange(-np.pi/2, np.pi/2 + y_width_phi*np.pi/180, y_width_phi*np.pi/180)
    
    # Create 2D histogram for phi
    H_phi, xedges_phi, yedges_phi = np.histogram2d(
        time_vals, df['phi_rad'], 
        bins=[n_time_bins, phi_bins],
        range=[time_range, None]
    )
    
    # Normalize by column (each time bin)
    H_phi_norm = H_phi.copy()
    for i in range(H_phi.shape[0]):
        col_sum = H_phi[i, :].sum()
        if col_sum > 0:
            H_phi_norm[i, :] = H_phi[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_phi_smooth = gaussian_filter(H_phi_norm, sigma=sigma)
    H_phi_smooth = np.ma.masked_where(H_phi_smooth < 0.001, H_phi_smooth)
    
    # Plot
    extent_phi = [xedges_phi[0], xedges_phi[-1], yedges_phi[0], yedges_phi[-1]]
    im_phi = ax_phi.imshow(H_phi_smooth.T, 
                           origin='lower',
                           aspect='auto',
                           extent=extent_phi,
                           cmap='magma',
                           interpolation='bilinear',
                           alpha=0.9)
    
    # Colorbar
    cbar_phi = plt.colorbar(im_phi, ax=ax_phi, pad=0.01)
    cbar_phi.set_label('Normalized Density', fontsize=10)
    
    # Format phi axis
    ax_phi.set_ylabel('Fast Direction φ (rad)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-np.pi/2, np.pi/2)
    ax_phi.axhline(0, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax_phi.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Set y-ticks in radians
    phi_ticks_rad = np.array([-np.pi/2, -np.pi/3, -np.pi/6, 0, np.pi/6, np.pi/3, np.pi/2])
    ax_phi.set_yticks(phi_ticks_rad)
    
    # Add eruption line (dashed white)
    ax_phi.axvline(0, color='white', linestyle='--', alpha=0.9, linewidth=2.5, 
                   label='Eruption Onset', zorder=10)
    ax_phi.legend(loc='upper right', fontsize=9, facecolor='white', 
                 edgecolor='white', framealpha=0.7)
    
    # === DT PLOT (in samples) ===
    dt_max_samples = df['dt_samples'].quantile(0.98)
    dt_bins = np.arange(0, min(40, dt_max_samples) + y_width_dt, y_width_dt)
    
    # Create 2D histogram for dt
    H_dt, xedges_dt, yedges_dt = np.histogram2d(
        time_vals, df['dt_samples'],
        bins=[n_time_bins, dt_bins],
        range=[time_range, None]
    )
    
    # Normalize by column
    H_dt_norm = H_dt.copy()
    for i in range(H_dt.shape[0]):
        col_sum = H_dt[i, :].sum()
        if col_sum > 0:
            H_dt_norm[i, :] = H_dt[i, :] / col_sum
    
    # Apply Gaussian smoothing
    H_dt_smooth = gaussian_filter(H_dt_norm, sigma=sigma)
    H_dt_smooth = np.ma.masked_where(H_dt_smooth < 0.001, H_dt_smooth)
    
    # Plot
    extent_dt = [xedges_dt[0], xedges_dt[-1], yedges_dt[0], yedges_dt[-1]]
    im_dt = ax_dt.imshow(H_dt_smooth.T,
                         origin='lower',
                         aspect='auto',
                         extent=extent_dt,
                         cmap='magma',
                         interpolation='bilinear',
                         alpha=0.9)
    
    # Colorbar
    cbar_dt = plt.colorbar(im_dt, ax=ax_dt, pad=0.01)
    cbar_dt.set_label('Normalized Density', fontsize=10)
    
    # Format dt axis
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Hours Relative to Eruption', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, 25)
    ax_dt.grid(True, alpha=0.3, linestyle='--', color='white')
    
    # Add secondary y-axis for seconds
    ax_dt_sec = ax_dt.secondary_yaxis('right', functions=(
        lambda x: x / sampling_rate,
        lambda x: x * sampling_rate
    ))
    ax_dt_sec.set_ylabel('δt (s)', fontsize=10)
    
    # Add eruption line (dashed white)
    ax_dt.axvline(0, color='white', linestyle='--', alpha=0.9, linewidth=2.5,
                  label='Eruption Onset', zorder=10)
    ax_dt.legend(loc='upper right', fontsize=9, facecolor='white',
                edgecolor='white', framealpha=0.7)
    
    # Format x-axis
    ax_dt.set_xlim(-hours_before, hours_after)
    ax_phi.set_xlim(-hours_before, hours_after)
    
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add title with statistics
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    title = (f"Splitting Parameters Around Eruption - Station {station}\n"
             f"Eruption: {eruption_time.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
             f"N = {len(df)} events | "
             f"φ: {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}° | "
             f"δt: {dt_mean_sec:.3f} ± {dt_std_sec:.3f} s")
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df

In [ ]:
fig, axes, df = plot_splitting_timeseries_smooth_before_after(
    results_swspy,
    qc_metrics_df,
    station='AXAS2, SWSPy',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    x_overlap = 0.95,
    y_width_dt=2,   # 2 samples
    sigma=2.0,
    eruption_time='2015-04-24T05:00:00Z'
)
plt.show()

In [ ]:
# Plot Baillard results around eruption
eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)
fig, axes, df_eruption_baillard = plot_splitting_timeseries_around_eruption(
    results_baillard,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2, Baillard',
    figsize=(4, 10),
    hours_before=24,
    hours_after=24,
    x_width_hours=2,
    y_width_phi=5,
    y_width_dt=2,
    sigma=2
)
plt.show()

In [ ]:
# Plot Baillard results around eruption
eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)
fig, axes, df_eruption_swpsy = plot_splitting_timeseries_around_eruption(
    results_swspy,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2, SWPSY',
    figsize=(4, 10),
    hours_before=24,
    hours_after=24,
    x_width_hours=2,
    y_width_phi=5,
    y_width_dt=2,
    sigma=2
)
plt.show()

In [ ]:
def plot_splitting_scatter_around_eruption(results_dict, qc_metrics_df, 
                                           eruption_time, station='AXAS2',
                                           hours_before=24, hours_after=24,
                                           figsize=(14, 10), window_hours=3,
                                           marker_size=30, marker_alpha=0.6,
                                           sampling_rate=200.0):
    """
    Create scatter plots of splitting parameters around eruption with rolling averages.
    Time axis shows hours relative to eruption (negative = before, positive = after).
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    qc_metrics_df : pd.DataFrame
        DataFrame with event metadata including origin_time
    eruption_time : str or pd.Timestamp
        Eruption onset time (e.g., "2015-04-24 05:00:00")
    station : str
        Station name for title
    hours_before : float
        Hours before eruption to include (default: 24)
    hours_after : float
        Hours after eruption to include (default: 24)
    figsize : tuple
        Figure size (width, height)
    window_hours : float
        Rolling window size in hours for moving average
    marker_size : float
        Size of scatter plot markers
    marker_alpha : float
        Transparency of scatter markers (0-1)
    sampling_rate : float
        Sampling rate in Hz (default 200 Hz)
    """
    import matplotlib.gridspec as gridspec
    
    # Convert eruption time to pandas Timestamp
    eruption_time = pd.to_datetime(eruption_time)
    
    # Define time window
    start_time = eruption_time - pd.Timedelta(hours=hours_before)
    end_time = eruption_time + pd.Timedelta(hours=hours_after)
    
    # Extract data from results within time window
    data_list = []
    for event_id, result in results_dict.items():
        if event_id in qc_metrics_df['event_id'].values:
            origin_time = pd.to_datetime(str(qc_metrics_df.loc[
                qc_metrics_df['event_id'] == event_id, 'origin_time'].values[0]))
            
            # Filter by time window
            if start_time <= origin_time <= end_time:
                phi = result['result']['phi']
                dt = result['result']['dt']
                
                # Calculate hours relative to eruption (negative = before)
                hours_from_eruption = (origin_time - eruption_time).total_seconds() / 3600.0
                
                # Convert dt from seconds to samples
                dt_samples = dt * sampling_rate
                
                data_list.append({
                    'hours_from_eruption': hours_from_eruption,
                    'phi_deg': phi,
                    'dt_samples': dt_samples,
                    'dt_seconds': dt
                })
    
    df = pd.DataFrame(data_list).sort_values('hours_from_eruption')
    
    if len(df) == 0:
        print(f"No data found in time window [{start_time} to {end_time}]")
        return None, None, None
    
    print(f"Found {len(df)} events in window ({hours_before}h before to {hours_after}h after eruption)")
    
    # Create figure with gridspec layout
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(2, 1, hspace=0.3, top=0.87, bottom=0.08, 
                          left=0.1, right=0.95)
    
    ax_phi = plt.subplot(gs[0])
    ax_dt = plt.subplot(gs[1], sharex=ax_phi)
    
    # Calculate rolling averages
    # Convert window size to number of events (approximate)
    time_span = df['hours_from_eruption'].max() - df['hours_from_eruption'].min()
    events_per_hour = len(df) / time_span if time_span > 0 else 1
    window_size = max(3, int(window_hours * events_per_hour))
    
    df['phi_rolling'] = df['phi_deg'].rolling(window=window_size, center=True, min_periods=1).mean()
    df['dt_rolling'] = df['dt_samples'].rolling(window=window_size, center=True, min_periods=1).mean()
    
    # === PHI SCATTER PLOT ===
    # Scatter plot
    ax_phi.scatter(df['hours_from_eruption'], df['phi_deg'], 
                   s=marker_size, alpha=marker_alpha, color='steelblue', 
                   edgecolors='black', linewidth=0.5, label='Individual measurements')
    
    # Rolling average
    ax_phi.plot(df['hours_from_eruption'], df['phi_rolling'], 
                color='red', linewidth=2.5, label=f'{window_hours}h rolling average', zorder=10)
    
    # Eruption line
    ax_phi.axvline(0, color='black', linestyle='--', linewidth=2, 
                   label='Eruption Onset', alpha=0.8, zorder=5)
    
    # Formatting
    ax_phi.set_ylabel('Fast Direction φ (°)', fontsize=12, fontweight='bold')
    ax_phi.set_ylim(-90, 90)
    ax_phi.axhline(0, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax_phi.grid(True, alpha=0.3, linestyle='--')
    ax_phi.legend(loc='upper right', fontsize=10, framealpha=0.9)
    
    # Add horizontal bands for reference
    ax_phi.axhspan(-90, -60, alpha=0.05, color='blue')
    ax_phi.axhspan(-30, 30, alpha=0.05, color='green')
    ax_phi.axhspan(60, 90, alpha=0.05, color='blue')
    
    # === DT SCATTER PLOT ===
    # Scatter plot
    ax_dt.scatter(df['hours_from_eruption'], df['dt_samples'], 
                  s=marker_size, alpha=marker_alpha, color='coral', 
                  edgecolors='black', linewidth=0.5, label='Individual measurements')
    
    # Rolling average
    ax_dt.plot(df['hours_from_eruption'], df['dt_rolling'], 
               color='darkred', linewidth=2.5, label=f'{window_hours}h rolling average', zorder=10)
    
    # Eruption line
    ax_dt.axvline(0, color='black', linestyle='--', linewidth=2, 
                  label='Eruption Onset', alpha=0.8, zorder=5)
    
    # Formatting
    ax_dt.set_ylabel('Delay Time δt (samples)', fontsize=12, fontweight='bold')
    ax_dt.set_xlabel('Hours Relative to Eruption', fontsize=12, fontweight='bold')
    ax_dt.set_ylim(0, max(40, df['dt_samples'].quantile(0.98) * 1.1))
    ax_dt.grid(True, alpha=0.3, linestyle='--')
    ax_dt.legend(loc='upper right', fontsize=10, framealpha=0.9)
    
    # Add secondary y-axis for seconds
    ax_dt_sec = ax_dt.secondary_yaxis('right', functions=(
        lambda x: x / sampling_rate,
        lambda x: x * sampling_rate
    ))
    ax_dt_sec.set_ylabel('δt (s)', fontsize=11)
    
    # Format x-axis
    ax_dt.set_xlim(-hours_before, hours_after)
    ax_phi.set_xlim(-hours_before, hours_after)
    
    # Remove x-labels from top plot
    plt.setp(ax_phi.xaxis.get_majorticklabels(), visible=False)
    
    # Add statistics text boxes
    phi_mean_deg = df['phi_deg'].mean()
    phi_std_deg = df['phi_deg'].std()
    dt_mean_sec = df['dt_seconds'].mean()
    dt_std_sec = df['dt_seconds'].std()
    
    # Split statistics by before/after eruption
    df_before = df[df['hours_from_eruption'] < 0]
    df_after = df[df['hours_from_eruption'] >= 0]
    
    stats_text = (
        f"All: φ = {phi_mean_deg:.1f}° ± {phi_std_deg:.1f}°, "
        f"δt = {dt_mean_sec:.3f} ± {dt_std_sec:.3f}s\n"
    )
    
    if len(df_before) > 0:
        phi_before = df_before['phi_deg'].mean()
        dt_before = df_before['dt_seconds'].mean()
        stats_text += f"Before: φ = {phi_before:.1f}°, δt = {dt_before:.3f}s (n={len(df_before)})\n"
    
    if len(df_after) > 0:
        phi_after = df_after['phi_deg'].mean()
        dt_after = df_after['dt_seconds'].mean()
        stats_text += f"After: φ = {phi_after:.1f}°, δt = {dt_after:.3f}s (n={len(df_after)})"
    
    # Add title
    title = (f"Splitting Parameters Around Eruption - Station {station}\n"
             f"Eruption: {eruption_time.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
             f"Total N = {len(df)} events\n"
             f"{stats_text}")
    fig.suptitle(title, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig, (ax_phi, ax_dt), df


# Example usage:
eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)

# Plot for SWSPy results
fig_swspy, axes_swspy, df_swspy = plot_splitting_scatter_around_eruption(
    results_swspy,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2 (SWSPy)',
    hours_before=24,
    hours_after=24,
    window_hours=3,  # 3-hour rolling average
    marker_size=40,
    marker_alpha=0.5
)
plt.show()

# Plot for Baillard results
fig_baillard, axes_baillard, df_baillard = plot_splitting_scatter_around_eruption(
    results_baillard_common,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2 (Baillard)',
    hours_before=24,
    hours_after=24,
    window_hours=3,
    marker_size=40,
    marker_alpha=0.5
)
plt.show()
